# Epsilon Fund - Walk-Forward Validation
Uses `infrastructure/walk_forward/` to run rolling Optuna optimisation and evaluate OOS robustness.

---

In [1]:
import sys
import os
import pandas as pd
import numpy as np


# ── Set your repo root path ────────────────────────────────────────────────────
ROOT = r'/Users/justiniturregui/Desktop/epsilon/github/Epsilon-Quant-Research'  # ← change to yours
# ──────────────────────────────────────────────────────────────────────────────

sys.path.append(ROOT + '/infrastructure/data')
sys.path.append(ROOT + '/infrastructure/walkforward')
sys.path.append(ROOT + '/infrastructure/backtester')


from binance_client import get_binance_client, get_data
from wf_engine import walk_forward, plateau_analysis, plateau_summary, perturbation_test, cost_stress_test
from wf_visualizer import plot_walk_forward_results, plot_plateau_analysis
from engine import backtest

---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: must be >= (train_bars + test_bars) * n_folds desired

In [2]:
SYMBOL   = 'BTCUSDT'
INTERVAL = '1d'
LOOKBACK = 2150 


# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client   = get_binance_client()
df = get_data(client, 'BTCUSDT', '1d', LOOKBACK)
print(f'Data: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
df.tail(3)

Data: 2020-05-12 → 2026-03-31  (2150 bars)


,Open,High,Low,Close,Volume
Time,,,,,
2026-03-29,66377.04,67130.50,65000.00,66010.93,10052.85584
2026-03-30,66010.93,68169.65,65800.59,66797.37,18353.38975
2026-03-31,66797.38,68408.37,65998.05,67450.00,13469.75815


---
## Parameter Configuration

Define which parameters to optimise and anchor - **recommended to do after strategy writeup**

`FIXED_PARAMS`: choose parameters with CV < 0.15 from prior stability run to reduce search space, improve OOS credibility



In [3]:
# ── parameter search space ────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
# Only keys present in PARAMS above are searched — remove a key from PARAMS to exclude it entirely.

PARAM_DEFS = {
    'ema_span':          ('int',   5,    40),
    'swing_caution':     ('int',   3,    14),
    'swing_stop':        ('int',   3,    10),
    'atr_caution':       ('int',   10,   30),
    'atr_stop':          ('int',   10,   30),
    'atr_size':          ('int',   3,    14),
    'caution_threshold': ('float', 0.8,  2.5),
    'adx_override':      ('int',   40,   80),
    'stop_atr_scale':    ('float', 0.5,  2.0),
    'risk_per_trade':    ('float', 0.005, 0.05),
    'max_leverage':      ('float', 1.0,  3.0),
    'stop_mult_pos_both':    ('float', 0.5, 2.5),
    'stop_mult_pos_caution': ('float', 0.1, 0.9),
    'stop_mult_pos_short':   ('float', 0.1, 1.5),
    'stop_mult_pos_normal':  ('float', 0.8, 2.0),
    'stop_mult_ent_both':    ('float', 0.5, 2.5),
    'stop_mult_ent_caution': ('float', 0.1, 0.9),
    'stop_mult_ent_short':   ('float', 0.1, 1.5),
    'stop_mult_ent_normal':  ('float', 0.5, 1.5),
    'vol_ma_period':  ('int',   10,  40),   # rolling window for volume MA
    'obv_ma_period':  ('int',   10,  40),   # OBV smoothing window
    'obv_lookback':   ('int',   10,  30),   # bars back to compare price vs OBV direction
}

# ── anchored params (set after a stability run; empty first time) ─────────────
# These bypass Optuna and are held constant across all folds.
# Populate using stability_df results: fix params with CV < 0.15
FIXED_PARAMS = {
'ema_span': 21,
'adx_override': 63,
'max_leverage': 3,
'stop_mult_ent_normal': 1,
'stop_mult_pos_normal': 1,
'risk_per_trade': 0.46,
'atr_size': 13,
    }

---
## Strategy

**CRUCIAL** - Strategy logic needs to work well in backtesting notebook before running here, saves time not running walk-forward for a broken strategy.

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> The engine applies a 1-bar execution lag automatically. Inside the strategy loop, use `prev` for signal decisions and `curr` for execution — no manual shifting needed.

**To implement your strategy:**
1. Write strategy logic — compute indicators, signals, and execution loop: use `param_name`for those to be searched
2. Update `indicator_cols` to list your longest-warmup indicators — the engine uses this to clean NaN rows after OOS trimming


In [4]:
def my_strategy(df_slice: pd.DataFrame, params: dict) -> pd.DataFrame:

    df = df_slice.copy()

    # ── strategy logic ────────────────────────────────────────────────────────
    df['EMA']          = df['Close'].ewm(span=params['ema_span'], adjust=False).mean()
    df['Swing_Hi_Cau'] = df['High'].rolling(params['swing_caution']).max()
    df['Swing_Lo_Cau'] = df['Low'].rolling(params['swing_caution']).min()
    df['Swing_Hi_Stp'] = df['High'].rolling(params['swing_stop']).max()

    def atr(period):
        hl = df['High'] - df['Low']
        hc = (df['High'] - df['Close'].shift(1)).abs()
        lc = (df['Low']  - df['Close'].shift(1)).abs()
        return pd.concat([hl, hc, lc], axis=1).max(axis=1).ewm(span=period, adjust=False).mean()

    df['ATR_Cau'] = atr(params['atr_caution'])
    df['ATR_Stp'] = atr(params['atr_stop'])
    df['ATR_Sz']  = atr(params['atr_size'])

    up    = df['High'].diff();  down = -df['Low'].diff()
    pdm   = up.where((up > down) & (up > 0), 0.0)
    ndm   = down.where((down > up) & (down > 0), 0.0)
    atr14 = atr(14)
    pdi   = 100 * pdm.ewm(span=14, adjust=False).mean() / atr14
    ndi   = 100 * ndm.ewm(span=14, adjust=False).mean() / atr14
    dx    = (100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, np.nan)).fillna(0)
    df['ADX_14'] = dx.ewm(span=14, adjust=False).mean()

    df['Vol_MA']  = df['Volume'].rolling(params['vol_ma_period']).mean()
    direction     = df['Close'].diff().apply(lambda x: 1 if x > 0 else -1)
    df['OBV']     = (df['Volume'] * direction).cumsum()
    df['OBV_MA']  = df['OBV'].rolling(params['obv_ma_period']).mean()


    df['Caution_OBV']   = (df['Close'] > df['Close'].shift(params['obv_lookback'])) & (df['OBV'] < df['OBV_MA'])
    df['Caution_Long']  = ((df['Swing_Hi_Cau'] - df['Low']) > params['caution_threshold'] * df['ATR_Cau']) | df['Caution_OBV']
    df['Caution_Short'] = ((df['High'] - df['Swing_Lo_Cau']) > params['caution_threshold'] * df['ATR_Cau']) | (df['Close'] > df['EMA'])
    df['Entry_Long'] = (df['Close'] > df['EMA']) & (~df['Caution_Long'] | (df['ADX_14'] > params['adx_override'])) & (df['Volume'] > df['Vol_MA'])
    df['position_size_raw'] = (params['risk_per_trade'] / (df['ATR_Sz'] / df['Close'])).clip(0.1, params['max_leverage'])

    n             = len(df)
    position      = [0]     * n
    position_size = [0.0]   * n
    stop_arr      = [np.nan] * n
    in_position   = 0
    stop_loss     = np.nan
    current_size  = 0.0

    for i in range(1, n):
        prev = df.iloc[i - 1]
        curr = df.iloc[i]

        if in_position == 0:
            if prev['Entry_Long']:
                in_position  = 1
                current_size = curr['position_size_raw']
                cl = prev['Caution_Long']; cs = prev['Caution_Short']
                if cl and cs: sm = params['stop_mult_ent_both']
                elif cl:        sm = params['stop_mult_ent_caution']
                else:           sm = params['stop_mult_ent_normal']
                stop_loss = curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale']
        else:
            if prev['Close'] < stop_loss:
                in_position  = 0
                current_size = 0.0
                stop_loss    = np.nan
            else:
                sm        = params['stop_mult_pos_caution'] if curr['Caution_Long'] else params['stop_mult_pos_normal']
                stop_loss = max(stop_loss, curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale'])

        position[i]      = in_position
        position_size[i] = current_size
        stop_arr[i]      = stop_loss

    df['position']      = position
    df['position_size'] = position_size
    df['stop_loss']     = stop_arr

    # ── cleanup ───────────────────────────────────────────────────────────────
    indicator_cols = ['EMA', 'ATR_Cau', 'ADX_14', 'Swing_Hi_Cau', 'Vol_MA', 'OBV_MA']
    df['position']      = df['position'].fillna(0).astype(int)
    df['position_size'] = df['position_size'].fillna(0.0)
    df['stop_loss']     = df['stop_loss'].fillna(0.0)

    return df, indicator_cols

---
## Run Walk-Forward
Simulates how the strategy would have performed if re-optimised periodically
in live trading, and exposes whether good IS performance survives unseen data.

**Folds Setup**
| Parameter | Description | Guidance |
|---|---|---|
| `TRAIN_BARS` | Bars per training window | Aim for 2:1 to 3:1 ratio vs `TEST_BARS` |
| `TEST_BARS` | Bars per test window | `365` = ~1 year on daily data |
| `BURNIN_BARS` | Bars prepended to test for indicator warmup | Match your longest indicator period |
| `N_TRIALS` | Optuna trials per fold | 300–500 for daily; more = better but slower |
| `COST` | Round-trip cost per trade | `0.001` = 0.1% |

**Score and Rejection** — use to calibrate what Optuna optimises IS: default `score_fn(m)` uses weighted basket of Sharpe, Calmar and Return, normalised using their "max" value; default `reject_fn(m)` discards runs failing certain criteria that limits credibility.

> Pay attention to the **degradation ratio** — OOS/IS Sharpe reveals overfitting.

In [5]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 1050  
TEST_BARS   = 275   
BURNIN_BARS = 100   
N_TRIALS    = 300  
COST        = 0.001 

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
# Modify weights or swap components. Must return a float (higher = better).

def score_fn(m):
    SHARPE_MAX = 2.5
    CALMAR_MAX = 70.0
    RETURN_MAX = 15.0

    calmar = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0

    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)

    return 0.50 * s + 0.30 * c + 0.20 * r

# ── REJECTION CRITERIA ────────────────────────────────────────────────────────
# Trials that return True are discarded (score → -999).

def reject_fn(m):
    if m is None:                      return True
    if m['num_trades']    < 20:        return True
    if m['win_rate']      < 0.35:      return True
    if m['max_drawdown']  < -0.8:     return True
    if m['profit_factor'] < 0.8:       return True
    return False


results = walk_forward(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,
    score_fn     = score_fn,    # ← your notebook definition
    reject_fn    = reject_fn,   # ← your notebook definition
    save_csv     = None,
)

Walk-forward: 4 fold(s)  train=1050  test=275  burnin=100  trials=300
  Fold 1: train 2020-05-12 → 2023-03-27  | test 2023-03-28 → 2023-12-27
  Fold 2: train 2021-02-11 → 2023-12-27  | test 2023-12-28 → 2024-09-27
  Fold 3: train 2021-11-13 → 2024-09-27  | test 2024-09-28 → 2025-06-29
  Fold 4: train 2022-08-15 → 2025-06-29  | test 2025-06-30 → 2026-03-31

Fixed (7): ['ema_span', 'adx_override', 'max_leverage', 'stop_mult_ent_normal', 'stop_mult_pos_normal', 'risk_per_trade', 'atr_size']
Free  (15): ['swing_caution', 'swing_stop', 'atr_caution', 'atr_stop', 'caution_threshold', 'stop_atr_scale', 'stop_mult_pos_both', 'stop_mult_pos_caution', 'stop_mult_pos_short', 'stop_mult_ent_both', 'stop_mult_ent_caution', 'stop_mult_ent_short', 'vol_ma_period', 'obv_ma_period', 'obv_lookback']

────────────────────────────────────────────────────────────
Fold 1/4  train: 2020-05-12 → 2023-03-27  test: 2023-03-28 → 2023-12-27


  0%|          | 0/300 [00:00<?, ?it/s]


  IS  → Sharpe: 1.78  Return: 4189.47%  DD: -58.35%  Calmar: 71.80  Trades: 43
  OOS → Sharpe: 1.44  Return: 79.51%  DD: -30.79%  Calmar: 2.58  Trades: 12

  Best params: {'ema_span': 21, 'adx_override': 63, 'max_leverage': 3, 'stop_mult_ent_normal': 1, 'stop_mult_pos_normal': 1, 'risk_per_trade': 0.46, 'atr_size': 13, 'swing_caution': 14, 'swing_stop': 9, 'atr_caution': 11, 'atr_stop': 13, 'caution_threshold': 1.685351218017504, 'stop_atr_scale': 1.8118038652644368, 'stop_mult_pos_both': 0.7802282943376816, 'stop_mult_pos_caution': 0.22456180034405446, 'stop_mult_pos_short': 0.2933558227644859, 'stop_mult_ent_both': 1.670007293669458, 'stop_mult_ent_caution': 0.8688512126328192, 'stop_mult_ent_short': 0.6229844556940134, 'vol_ma_period': 12, 'obv_ma_period': 24, 'obv_lookback': 18}

────────────────────────────────────────────────────────────
Fold 2/4  train: 2021-02-11 → 2023-12-27  test: 2023-12-28 → 2024-09-27


  0%|          | 0/300 [00:00<?, ?it/s]


  IS  → Sharpe: 1.24  Return: 330.23%  DD: -29.77%  Calmar: 11.09  Trades: 65
  OOS → Sharpe: 2.09  Return: 92.79%  DD: -14.48%  Calmar: 6.41  Trades: 16

  Best params: {'ema_span': 21, 'adx_override': 63, 'max_leverage': 3, 'stop_mult_ent_normal': 1, 'stop_mult_pos_normal': 1, 'risk_per_trade': 0.46, 'atr_size': 13, 'swing_caution': 7, 'swing_stop': 10, 'atr_caution': 20, 'atr_stop': 27, 'caution_threshold': 1.4268284644862792, 'stop_atr_scale': 0.5033682289540878, 'stop_mult_pos_both': 0.8352196097546601, 'stop_mult_pos_caution': 0.15353282422516207, 'stop_mult_pos_short': 0.4862098619326445, 'stop_mult_ent_both': 1.6456011645022035, 'stop_mult_ent_caution': 0.7745313384273218, 'stop_mult_ent_short': 0.4204391896198423, 'vol_ma_period': 15, 'obv_ma_period': 10, 'obv_lookback': 16}

────────────────────────────────────────────────────────────
Fold 3/4  train: 2021-11-13 → 2024-09-27  test: 2024-09-28 → 2025-06-29


  0%|          | 0/300 [00:00<?, ?it/s]


  IS  → Sharpe: 1.47  Return: 1096.31%  DD: -69.17%  Calmar: 15.85  Trades: 33
  OOS → Sharpe: 1.56  Return: 98.84%  DD: -30.52%  Calmar: 3.24  Trades: 11

  Best params: {'ema_span': 21, 'adx_override': 63, 'max_leverage': 3, 'stop_mult_ent_normal': 1, 'stop_mult_pos_normal': 1, 'risk_per_trade': 0.46, 'atr_size': 13, 'swing_caution': 6, 'swing_stop': 8, 'atr_caution': 10, 'atr_stop': 13, 'caution_threshold': 2.0421724194663438, 'stop_atr_scale': 1.8419918041939805, 'stop_mult_pos_both': 0.6394081265316868, 'stop_mult_pos_caution': 0.5837824851005398, 'stop_mult_pos_short': 0.6309361037021718, 'stop_mult_ent_both': 1.012777999060069, 'stop_mult_ent_caution': 0.6644938697723691, 'stop_mult_ent_short': 0.5347700014034962, 'vol_ma_period': 39, 'obv_ma_period': 32, 'obv_lookback': 17}

────────────────────────────────────────────────────────────
Fold 4/4  train: 2022-08-15 → 2025-06-29  test: 2025-06-30 → 2026-03-31


  0%|          | 0/300 [00:00<?, ?it/s]


  IS  → Sharpe: 1.82  Return: 3624.27%  DD: -55.26%  Calmar: 65.59  Trades: 37
  OOS → Sharpe: -2.22  Return: -60.30%  DD: -60.30%  Calmar: -1.00  Trades: 9

  Best params: {'ema_span': 21, 'adx_override': 63, 'max_leverage': 3, 'stop_mult_ent_normal': 1, 'stop_mult_pos_normal': 1, 'risk_per_trade': 0.46, 'atr_size': 13, 'swing_caution': 3, 'swing_stop': 6, 'atr_caution': 16, 'atr_stop': 17, 'caution_threshold': 2.4229673530798506, 'stop_atr_scale': 1.8344387384429561, 'stop_mult_pos_both': 2.3169134078458287, 'stop_mult_pos_caution': 0.7224262012530512, 'stop_mult_pos_short': 0.8266255113208165, 'stop_mult_ent_both': 2.4152112886440937, 'stop_mult_ent_caution': 0.8739117158170127, 'stop_mult_ent_short': 1.1913249986385672, 'vol_ma_period': 35, 'obv_ma_period': 29, 'obv_lookback': 11}

════════════════════════════════════════════════════════════
WALK-FORWARD SUMMARY
════════════════════════════════════════════════════════════

Out-of-sample across 4 fold(s):
  Avg Sharpe:       0.72
 

---
## Granular Results and Parameter Stability

Per-fold IS vs OOS performance. Each row is one fold — compare `train_*` vs `test_*` columns to assess overfitting.

| Column | Description |
|---|---|
| `*_sharpe` `*_return` `*_drawdown` `*_calmar` | Core performance metrics |
| `*_trades` `*_winrate` `*_profit_factor` | Trade statistics |
| `optuna_score` | Best score achieved on training window |
| `param_*` | Best parameter values per fold e.g. `param_ema_span` |

**Concensus Parameters** - use to anchor: the engine determines stability using the coefficient of variation (CV) — the standard deviation of a parameter's best values across all folds divided by their median.

>CV < 0.15: indicates the strategy  relies on value rather than it being noise-fitted to a specific period — making it safe to fix for future runs. A high CV means the parameter is period-sensitive and should stay free.

In [6]:
# ── fold summary table ────────────────────────────────────────────────────────
display_cols = [
    'train_start', 'train_end',
    'test_start', 'test_end',
    'train_sharpe', 'test_sharpe',
    'train_return', 'test_return',
    'train_drawdown', 'test_drawdown',
    'test_trades',  'test_winrate', 'optuna_score',
]
display(results['results_df'][display_cols].round(2))

# ── parameter stability ───────────────────────────────────────────────────────
stability = results['stability_df'].copy()
stability['stable'] = stability['stable'].map({True: '✓', False: ''})
stability['fixed']  = stability['fixed'].map({True: '✓', False: ''})
stability = stability[['param', 'median', 'std', 'cv', 'stable', 'fixed']].round(2)
display(stability.sort_values('cv'))

# ── consensus params ──────────────────────────────────────────────────────────
stable = results['stability_df'][results['stability_df']['cv'] < 0.15]

print('Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:')
for _, row in stable.iterrows():
    v = results['consensus_params'][row['param']]
    v_fmt = int(round(v)) if isinstance(v, float) and v == int(v) else round(v, 4) if isinstance(v, float) else v
    print(f"    '{row['param']}': {v_fmt},")
    
print('\nConsensus parameters (median across folds):')
for k, v in results['consensus_params'].items():
    print(f'  {k:<30} = {round(v, 2) if isinstance(v, float) else v}')

,train_start,train_end,test_start,test_end,train_sharpe,test_sharpe,train_return,test_return,train_drawdown,test_drawdown,test_trades,test_winrate,optuna_score
0,2020-05-12,2023-03-27,2023-03-28,2023-12-27,1.78,1.44,41.89,0.80,-0.58,-0.31,12,0.42,0.86
1,2021-02-11,2023-12-27,2023-12-28,2024-09-27,1.24,2.09,3.30,0.93,-0.30,-0.14,16,0.69,0.34
2,2021-11-13,2024-09-27,2024-09-28,2025-06-29,1.47,1.56,10.96,0.99,-0.69,-0.31,11,0.55,0.51
3,2022-08-15,2025-06-29,2025-06-30,2026-03-31,1.82,-2.22,36.24,-0.60,-0.55,-0.60,9,0.11,0.84


,param,median,std,cv,stable,fixed
0,ema_span,21.00,0.00,0.00,✓,✓
18,stop_mult_ent_normal,1.00,0.00,0.00,✓,✓
14,stop_mult_pos_normal,1.00,0.00,0.00,✓,✓
9,risk_per_trade,0.46,0.00,0.00,✓,✓
7,adx_override,63.00,0.00,0.00,✓,✓
10,max_leverage,3.00,0.00,0.00,✓,✓
5,atr_size,13.00,0.00,0.00,✓,✓
16,stop_mult_ent_caution,0.82,0.09,0.10,✓,
21,obv_lookback,16.50,2.69,0.16,,
2,swing_stop,8.50,1.48,0.17,,


Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:
    'ema_span': 21,
    'atr_size': 13,
    'adx_override': 63,
    'risk_per_trade': 0.46,
    'max_leverage': 3,
    'stop_mult_pos_normal': 1,
    'stop_mult_ent_caution': 0.8217,
    'stop_mult_ent_normal': 1,

Consensus parameters (median across folds):
  ema_span                       = 21
  swing_caution                  = 6
  swing_stop                     = 8
  atr_caution                    = 14
  atr_stop                       = 15
  atr_size                       = 13
  caution_threshold              = 1.86
  adx_override                   = 63
  stop_atr_scale                 = 1.82
  risk_per_trade                 = 0.46
  max_leverage                   = 3.0
  stop_mult_pos_both             = 0.81
  stop_mult_pos_caution          = 0.4
  stop_mult_pos_short            = 0.56
  stop_mult_pos_normal           = 1.0
  stop_mult_ent_both             = 1.66
  stop_mult_ent_caution          = 0.82
  stop_mult_ent_short   

---
## Parameter Robustness Checks

### Plateau Analysis
Sweep each free parameter across its range while holding others at consensus (median) value then evaluates the `score` at each point by backtesting over entire lookback.

The stability table (CV across folds) tells you *"does the optimizer consistently pick the same value?"*  

Plateau analysis tells you *"if that value were slightly wrong, would performance collapse?"*  

**Plateau %** - what fraction of each parameter's range stays within 20% of peak score: >60% = `robust plateau`, 30–60% = `moderate`, <30% = `fragile` (consider anchoring). `N/A` means every sweep point failed rejection filters — the strategy is completely intolerant of movement on that dimension.

>Run time: `n_free_params` × `n_steps`

|  | Robust Plateau| Fragile Plateau |
|--|-------------------|-------------------|
| Low CV | Stable across folds and insensitive to exact value | Looks stable but is fitting the same noise patterns - fix at concensus|
| High CV | Parameter unimportant - fix at any reasonable value | Unstable across folds and sitting on a cliff - strong candidate to eliminate |


### 1-D sweep charts:
| Element | Meaning | Good | Bad |
|---------|---------|------|-----|
| **Blue curve** | Composite score at each value of the parameter, with all others held at consensus | Flat-topped curve — performance is insensitive to the exact value | Narrow spike — optimizer latched onto one specific value, everything nearby is worse |
| **Red dashed line** | Where the consensus value sits | On the flat top of the curve | On a steep slope or at the edge of a cliff |
| **Green dashed line** | Cutoff at 80% of peak score — the boundary between plateau and non-plateau | Blue curve stays above this line across most of the range | Blue curve dips below it quickly either side of the peak |
| **Green shading** | Plateau region — all values where the score stays within 20% of the peak | Wide green band spanning most of the range (robust) | Thin sliver or no shading at all (fragile/overfit) |

### Perturbation test
Jitters all free parameters by ±5/10/20% of their range (50 random samples per offset range). Measures how much the score degrades vs the base

Test whether optimum is a broad hill in `#free params`-D space or a narrow spike

**>15%:** fragile optimum, consider reducing free parameters

In [7]:
# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
sweep_results = plateau_analysis(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    n_steps      = 20, #Adjust for number of steps around concensus per parameter
)

# ── text verdicts ──────────────────────────────────────────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params = results['consensus_params'],
    threshold   = 0.20, #Adjust for % around peak score
)

# ── visual sweep curves ───────────────────────────────────────────────────
plot_plateau_analysis(
    sweep_results    = sweep_results,
    consensus_params = results['consensus_params'],
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    threshold        = 0.20,
    show             = False,
    save_html        = None,
)


# ── neighbouahood perturbation ────────────────────────────────────────────
# Randomly jitters ALL free params simultaneously.
# If mean score degrades >15% at ±10% offset, the optimum is fragile.

perturb_df = perturbation_test(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    pct_offsets  = (0.05, 0.10, 0.20),   # ±5%, ±10%, ±20% of range
    n_samples    = 50,                     # random perturbations per offset level
)


═══════════════════════════════════════════════════════════════════════════
PLATEAU ANALYSIS — PARAMETER ROBUSTNESS
═══════════════════════════════════════════════════════════════════════════
Parameter                          Base     Peak  Plateau%  Verdict
────────────────────────────── ──────── ──────── ─────────  ────────────────────
swing_caution                       N/A      N/A       N/A  no valid trials
swing_stop                          N/A      N/A       N/A  no valid trials
atr_caution                         N/A      N/A       N/A  no valid trials
atr_stop                            N/A      N/A       N/A  no valid trials
caution_threshold                1.8638    0.484     25.0%  FRAGILE — consider fixing
stop_atr_scale                   1.8231    0.190    100.0%  robust plateau
stop_mult_pos_both                  N/A      N/A       N/A  no valid trials
stop_mult_pos_caution               N/A      N/A       N/A  no valid trials
stop_mult_pos_short                 N/A  

---
## Results Charts and Cost Stress Test

| Parameter | Description | Default |
|---|---|---|
| `show_fold_perf` | IS vs OOS bars for return, Sharpe, drawdown per fold | `False` |
| `show_param_evol` | Parameter evolution across folds with ±1 std bands | `False` |
| `show_oos_equity` | Combined OOS equity curve + drawdown with fold boundaries | `True` |
| `show_trades` | Overlay entry/exit markers on OOS equity chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | `None` |
| `save_html_dir` | Directory path to save charts as HTML files, or `None` | `None` |

**Cost Stress Test:** re-run the combined OOS backtest at 1×, 1.5×, 2×, 3× the base cost. Fragile strategies collapse; robust ones degrade gradually.

In [8]:
plot_walk_forward_results(
    results          = results,
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    benchmark_data   = df,
    show             = True,
    save_html_dir    = None,
    show_fold_perf   = False,   # IS vs OOS bars by fold
    show_param_evol  = False,   # parameter evolution across folds
    show_oos_equity  = True,   # combined OOS equity curve
    show_trades      = False,  # trade markers on OOS equity chart
)

# ── transaction cost stress test ──────────────────────────────────────────

if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test')


═══════════════════════════════════════════════════════════════════════════
TRANSACTION COST STRESS TEST
═══════════════════════════════════════════════════════════════════════════
    Cost   Mult   Sharpe     Return      MaxDD   Calmar       PF
──────── ────── ──────── ────────── ────────── ──────── ────────
  0.0010   1.0x     0.84    173.19%    -72.42%     2.39     1.82
  0.0015   1.5x     0.81    160.50%    -72.71%     2.21     1.82
  0.0020   2.0x     0.78    148.39%    -73.00%     2.03     1.82
  0.0030   3.0x     0.73    125.81%    -73.57%     1.71     1.82
